In [ ]:
# Εγκατάσταση και εισαγωγή βιβλιοθηκών

# Εγκατάσταση του πακέτου pycaret
# Προκειμένου να αποφευχθεί η ασταθής και χρονοβόρα διαδικασία υποβάθμισης (downgrade) του πυρήνα της Python στο Colab, επιλέχθηκε η απευθείας εγκατάσταση της έκδοσης ανάπτυξης (master branch) από το επίσημο αποθετήριο του PyCaret στο GitHub
!pip install git+https://github.com/pycaret/pycaret.git@master --upgrade

# ΣΗΜΕΙΩΣΗ: Μετά την εγκατάσταση του PyCaret, το Colab ίσως ζητήσει "Restart Session".
# Αν εμφανιστεί σχετικό κουμπί (Restart Runtime/Session), πατήστε το και μετά προχωρήστε στο επόμενο κελί.

# Εισαγωγή της βιβλιοθήκης pandas για τη διαχείριση των δεδομένων
import pandas as pd

# Εισαγωγή της βιβλιοθήκης numpy για να εντοπίσουμε τις «άπειρες» τιμές (infinity) στο dataset
import numpy as np

In [ ]:
# Φόρτωση των δεδομένων

# Το link του CSV αρχείου που περιέχει τα δεδομένα DarkNet
csv_url = "https://raw.githubusercontent.com/jimtak80/AI_EKPA/refs/heads/main/Data/DarkNet.csv"

# Φόρτωση του συνόλου δεδομένων (dataset) σε ένα pandas DataFrame
df = pd.read_csv(csv_url)

# Εμφάνιση των πρώτων 5 γραμμών του dataset για να επιβεβαιώσουμε ότι φορτώθηκε σωστά
display(df.head())

# Εμφάνιση των διαστάσεων του dataset (γραμμές, στήλες)
print(f"\nΤο dataset έχει {df.shape[0]} γραμμές και {df.shape[1]} στήλες.")

In [ ]:
# Διερευνητική Ανάλυση Δεδομένων (EDA)

# 1. Γενικές πληροφορίες για το dataset (τύποι μεταβλητών, πλήθος εγγραφών, memory usage)
print("--- 1. Γενικές Πληροφορίες Dataset ---")
df.info()

# 2. Στατιστική σύνοψη των αριθμητικών μεταβλητών (μέσος όρος, τυπική απόκλιση, min, max)
print("\n--- 2. Στατιστική Σύνοψη ---")
display(df.describe())

# 3. Έλεγχος ελλιπών (κενών) τιμών
total_nans = df.isnull().sum().sum()
print(f"\n--- 3. Έλεγχος Κενών Τιμών ---")
print(f"Συνολικές κενές τιμές στο dataset: {total_nans}")

# 4. Έλεγχος της στήλης Στόχου (Label)
target_col = df.columns[-1] # Αποθηκεύουμε το όνομα της τελευταίας στήλης
print(f"\n--- 4. Ανάλυση Στήλης Στόχου: '{target_col}' ---")

# Εμφανίζουμε πόσες εγγραφές υπάρχουν για κάθε κατηγορία (π.χ. πόσα Tor και πόσα Non-Tor)
# Αυτό μας βοηθάει να δούμε αν το dataset μας είναι ισορροπημένο (balanced)
label_counts = df[target_col].value_counts()
display(label_counts)

In [ ]:
# ΓΡΗΓΟΡΗ Αρχικοποίηση PyCaret & Σύγκριση Επιλεγμένων Μοντέλων

from pycaret.classification import setup, compare_models

print("-> 0. Καθαρισμός δεδομένων (αφαίρεση άπειρων τιμών)...")
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

target_col = df.columns[-1]
print(f"-> Η στήλη - στόχος (Label) είναι η: '{target_col}'\n")

# 1. Setup του περιβάλλοντος
print("-> 1. Γίνεται η αρχικοποίηση (Setup) του PyCaret...")
clf_setup = setup(
    data = df,
    target = target_col,
    session_id = 42,
    normalize = True
)

print("\n-> Το Setup ολοκληρώθηκε! Ξεκινάει η ΓΡΗΓΟΡΗ σύγκριση...")

# 2. ΓΡΗΓΟΡΗ ΕΚΠΑΙΔΕΥΣΗ:
# include: Δοκιμάζουμε μόνο 5 επιλεγμένα μοντέλα (dt, rf, lightgbm, lr, nb)
# fold=3: Μειώνουμε τις επαναλήψεις εκπαίδευσης από 10 σε 3
best_model = compare_models(include=['dt', 'rf', 'lightgbm', 'lr', 'nb'], fold=3)

# 3. Εμφάνιση του καλύτερου
print("\n" + "="*50)
print("ΤΟ ΚΑΛΥΤΕΡΟ ΜΟΝΤΕΛΟ ΠΟΥ ΒΡΕΘΗΚΕ ΕΙΝΑΙ:")
print("="*50)
print(best_model)

In [ ]:
# Αξιολόγηση και Οπτικοποίηση του Καλύτερου Μοντέλου

from pycaret.classification import plot_model, predict_model

print("-> 1. Δημιουργία Γραφημάτων (Αξιολόγηση)...\n")

# Γράφημα 1: Μήτρα Σύγχυσης (Confusion Matrix)
# Δείχνει πόσες σωστές και πόσες λάθος προβλέψεις έκανε το μοντέλο για κάθε κατηγορία.
print("--- Confusion Matrix (Μήτρα Σύγχυσης) ---")
plot_model(best_model, plot='confusion_matrix')

# Γράφημα 2: Σημαντικότητα Χαρακτηριστικών (Feature Importance)
# Δείχνει ποιες στήλες/χαρακτηριστικά της κίνησης έπαιξαν τον μεγαλύτερο ρόλο στην πρόβλεψη.
print("--- Feature Importance (Σημαντικότητα Χαρακτηριστικών) ---")
plot_model(best_model, plot='feature')

# Γράφημα 3: Καμπύλη ROC (AUC)
# Οπτικοποιεί την ικανότητα του μοντέλου να ξεχωρίζει τις κλάσεις (Tor vs Non-Tor).
print("--- ROC Curve ---")
plot_model(best_model, plot='auc')

print("\n-> 2. Πρόβλεψη σε νέα 'άγνωστα' δεδομένα (Test Set)...")
# Το μοντέλο δοκιμάζεται στα δεδομένα που κράτησε αυτόματα το PyCaret για testing.
predictions = predict_model(best_model)

print("\nΤα αποτελέσματα των προβλέψεων (τελευταίες 2 στήλες: 'prediction_label' και 'prediction_score'):")
display(predictions.head(10))

In [ ]:
# Αποθήκευση (Save) του Μοντέλου

from pycaret.classification import save_model, load_model

print("-> Ξεκινάει η αποθήκευση του μοντέλου...")

# Αποθηκεύουμε το μοντέλο με ένα αναγνωρίσιμο όνομα
# Το PyCaret θα δημιουργήσει αυτόματα ένα αρχείο με κατάληξη '.pkl' (Pickle file)
save_model(best_model, 'Final_RF_DarkNet_Model')

print("\n-> Το μοντέλο αποθηκεύτηκε επιτυχώς!")
print("-> (Θα το βρεις στα αριστερά της οθόνης, στο εικονίδιο με τον Φάκελο - Files. Λέγεται 'Final_RF_DarkNet_Model.pkl')")

# --- Προαιρετική Επιβεβαίωση ---
# Ας δοκιμάσουμε να το φορτώσουμε ξανά για να βεβαιωθούμε ότι δουλεύει σωστά
print("\n-> Επιβεβαίωση: Φόρτωση του αποθηκευμένου μοντέλου...")
loaded_model = load_model('Final_RF_DarkNet_Model')
print("-> Το μοντέλο φορτώθηκε με επιτυχία! Το σύστημα είναι έτοιμο για Threat Hunting!")